# 09 annotation bias qc

**Purpose:** Audit annotation composition and candidate CNV-reference groups.

**Original analysis notebook:** `09_annotation_bias_composition_QC_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 25 — Annotation bias / composition QC before CNV

This notebook evaluates whether the marker-based annotations and clusters are biologically interpretable or driven by dataset, condition, patient, or sample composition.

Main purpose:

- Keep the full-gene object as the authoritative object.
- Preserve `adata.layers["counts"]`.
- Add cluster-level bias/review/CNV-role columns to `adata.obs`.
- Save a stage-specific h5ad for the next notebooks.
- Generate TSV reports and QC figures.

Main cluster key:

```text
leiden_scvi_main
```

which corresponds to:

```text
leiden_scvi_HVG8000_r0_6
```


In [ ]:
# =========================
# Imports and paths
# =========================

from pathlib import Path
import warnings
import gc

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOG_DIR = PROJECT_DIR / "logs"

INTEGRATED_DIR = PROCESSED_DIR / "integrated"
FIG_DIR = FIGURES_DIR / "annotation_bias_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATED_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"

ANNOTATION_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6.tsv"
EVIDENCE_TSV = METADATA_DIR / "cluster_marker_annotation_evidence_HVG8000_r0_6.tsv"
ANNOTATION_COUNTS_TSV = METADATA_DIR / "annotation_marker_based_HVG8000_r0_6_cell_counts.tsv"

OUT_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"
OUT_ALIAS_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_annotation_bias_ready.h5ad"

CLUSTER_KEY = "leiden_scvi_main"
PRIMARY_CLUSTER_KEY = "leiden_scvi_HVG8000_r0_6"

VERSION_TAG = "HVG8000_r0_6"

# Output TSVs
READINESS_TSV = METADATA_DIR / f"annotation_bias_readiness_report_{VERSION_TAG}.tsv"
CLUSTER_SUMMARY_TSV = METADATA_DIR / f"annotation_bias_cluster_summary_{VERSION_TAG}.tsv"
COMPOSITION_LONG_TSV = METADATA_DIR / f"annotation_bias_composition_long_{VERSION_TAG}.tsv"
BIAS_FLAGS_TSV = METADATA_DIR / f"annotation_bias_flags_{VERSION_TAG}.tsv"
CNV_ROLE_TSV = METADATA_DIR / f"annotation_cnv_role_recommendations_{VERSION_TAG}.tsv"
CELL_COUNTS_BY_STATUS_TSV = METADATA_DIR / f"annotation_bias_cell_counts_by_status_{VERSION_TAG}.tsv"
DECISION_REPORT_TSV = METADATA_DIR / f"annotation_bias_decision_report_{VERSION_TAG}.tsv"
OUTPUT_MANIFEST_TSV = METADATA_DIR / f"annotation_bias_output_manifest_{VERSION_TAG}.tsv"


def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def safe_str_series(s):
    return s.astype(str).replace({"nan": "unknown", "None": "unknown"}).fillna("unknown")


def find_first_existing_column(columns, candidates):
    columns = list(columns)
    for c in candidates:
        if c in columns:
            return c
    return None


def detect_column(columns, must_include=None, must_not_include=None, fallback=None):
    must_include = [x.lower() for x in (must_include or [])]
    must_not_include = [x.lower() for x in (must_not_include or [])]
    for col in columns:
        low = str(col).lower()
        if all(x in low for x in must_include) and not any(x in low for x in must_not_include):
            return col
    return fallback


print("Project:", PROJECT_DIR)
print("Input h5ad:", ANNOTATED_H5AD)
print("Input exists:", ANNOTATED_H5AD.exists())


In [ ]:
# =========================
# Read annotated full-gene object and validate
# =========================

if not ANNOTATED_H5AD.exists():
    raise FileNotFoundError(f"Annotated h5ad not found: {ANNOTATED_H5AD}")

adata = sc.read_h5ad(ANNOTATED_H5AD)
print(adata)

if CLUSTER_KEY not in adata.obs.columns:
    if PRIMARY_CLUSTER_KEY in adata.obs.columns:
        adata.obs[CLUSTER_KEY] = adata.obs[PRIMARY_CLUSTER_KEY].astype(str)
        print(f"[ADDED] {CLUSTER_KEY} from {PRIMARY_CLUSTER_KEY}")
    else:
        raise KeyError(f"Neither {CLUSTER_KEY} nor {PRIMARY_CLUSTER_KEY} found in adata.obs.")

adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str)

# Detect annotation columns already present in obs
obs_cols = list(adata.obs.columns)

annotation_col = detect_column(
    obs_cols,
    must_include=["annotation", "hvg8000"],
    must_not_include=["confidence", "note", "review", "bias", "count", "composition"],
)
lineage_col = detect_column(
    obs_cols,
    must_include=["lineage", "hvg8000"],
)
confidence_col = detect_column(
    obs_cols,
    must_include=["confidence", "hvg8000"],
)
note_col = detect_column(
    obs_cols,
    must_include=["note", "hvg8000"],
)

# Fallbacks if exact version-tagged names are not found
if annotation_col is None:
    annotation_col = detect_column(
        obs_cols,
        must_include=["annotation"],
        must_not_include=["confidence", "note", "review", "bias", "count", "composition"],
    )
if lineage_col is None:
    lineage_col = detect_column(obs_cols, must_include=["lineage"])
if confidence_col is None:
    confidence_col = detect_column(obs_cols, must_include=["confidence"])
if note_col is None:
    note_col = detect_column(obs_cols, must_include=["note"])

print("Detected annotation_col:", annotation_col)
print("Detected lineage_col:", lineage_col)
print("Detected confidence_col:", confidence_col)
print("Detected note_col:", note_col)

# If annotation columns are missing, reconstruct them from annotation TSV
if annotation_col is None and ANNOTATION_TSV.exists():
    ann = pd.read_csv(ANNOTATION_TSV, sep="\t")
    ann.columns = [str(c) for c in ann.columns]
    print("[READ]", ANNOTATION_TSV, ann.shape)
    print("Annotation TSV columns:", list(ann.columns))

    cluster_col_ann = find_first_existing_column(ann.columns, ["cluster", CLUSTER_KEY, PRIMARY_CLUSTER_KEY])
    if cluster_col_ann is None:
        raise KeyError("No cluster column found in annotation TSV.")
    ann[cluster_col_ann] = ann[cluster_col_ann].astype(str)

    ann_annotation_col = detect_column(
        ann.columns,
        must_include=["annotation"],
        must_not_include=["confidence", "note", "count", "composition"],
    )
    ann_lineage_col = detect_column(ann.columns, must_include=["lineage"])
    ann_confidence_col = detect_column(ann.columns, must_include=["confidence"])
    ann_note_col = detect_column(ann.columns, must_include=["note"])

    annotation_col = f"annotation_marker_based_{VERSION_TAG}"
    lineage_col = f"lineage_marker_based_{VERSION_TAG}"
    confidence_col = f"annotation_confidence_marker_based_{VERSION_TAG}"
    note_col = f"annotation_note_marker_based_{VERSION_TAG}"

    if ann_annotation_col:
        adata.obs[annotation_col] = adata.obs[CLUSTER_KEY].map(dict(zip(ann[cluster_col_ann], ann[ann_annotation_col]))).fillna("unknown")
    else:
        adata.obs[annotation_col] = "unknown"

    if ann_lineage_col:
        adata.obs[lineage_col] = adata.obs[CLUSTER_KEY].map(dict(zip(ann[cluster_col_ann], ann[ann_lineage_col]))).fillna("unknown")
    else:
        adata.obs[lineage_col] = "unknown"

    if ann_confidence_col:
        adata.obs[confidence_col] = adata.obs[CLUSTER_KEY].map(dict(zip(ann[cluster_col_ann], ann[ann_confidence_col]))).fillna("unknown")
    else:
        adata.obs[confidence_col] = "unknown"

    if ann_note_col:
        adata.obs[note_col] = adata.obs[CLUSTER_KEY].map(dict(zip(ann[cluster_col_ann], ann[ann_note_col]))).fillna("")
    else:
        adata.obs[note_col] = ""

for col in [annotation_col, lineage_col, confidence_col, note_col]:
    if col is not None and col in adata.obs.columns:
        adata.obs[col] = safe_str_series(adata.obs[col])

required_metadata = ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]
for col in required_metadata:
    if col not in adata.obs.columns:
        print(f"[WARNING] Missing metadata column: {col}. Creating as unknown.")
        adata.obs[col] = "unknown"
    adata.obs[col] = safe_str_series(adata.obs[col])

readiness = pd.DataFrame([
    {"metric": "n_cells", "value": adata.n_obs},
    {"metric": "n_genes", "value": adata.n_vars},
    {"metric": "cluster_key", "value": CLUSTER_KEY},
    {"metric": "n_clusters", "value": adata.obs[CLUSTER_KEY].nunique()},
    {"metric": "annotation_col", "value": annotation_col},
    {"metric": "lineage_col", "value": lineage_col},
    {"metric": "confidence_col", "value": confidence_col},
    {"metric": "note_col", "value": note_col},
    {"metric": "counts_layer_present", "value": "counts" in adata.layers},
    {"metric": "X_scVI_present", "value": "X_scVI" in adata.obsm},
    {"metric": "X_umap_present", "value": "X_umap" in adata.obsm},
])

write_tsv_replace(readiness, READINESS_TSV)
display(readiness)


In [ ]:
# =========================
# Cluster composition summaries
# =========================

def entropy_from_counts(counts):
    arr = np.asarray(counts, dtype=float)
    total = arr.sum()
    if total <= 0:
        return 0.0
    p = arr / total
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def normalized_entropy_from_counts(counts):
    arr = np.asarray(counts, dtype=float)
    k = (arr > 0).sum()
    if k <= 1:
        return 0.0
    return float(entropy_from_counts(arr) / np.log2(k))


def top_category_info(df, cluster_col, category_col):
    tab = (
        df.groupby([cluster_col, category_col], observed=True)
        .size()
        .reset_index(name="n")
    )
    total = tab.groupby(cluster_col)["n"].transform("sum")
    tab["fraction"] = tab["n"] / total
    idx = tab.groupby(cluster_col)["fraction"].idxmax()
    top = tab.loc[idx].copy()
    top = top.rename(columns={
        category_col: f"top_{category_col}",
        "n": f"top_{category_col}_n",
        "fraction": f"top_{category_col}_fraction",
    })
    return top[[cluster_col, f"top_{category_col}", f"top_{category_col}_n", f"top_{category_col}_fraction"]]


def n_unique_by_cluster(df, cluster_col, category_col):
    out = (
        df.groupby(cluster_col, observed=True)[category_col]
        .nunique()
        .reset_index(name=f"n_{category_col}")
    )
    return out


obs_df = adata.obs[
    [CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]
].copy()

obs_df[CLUSTER_KEY] = obs_df[CLUSTER_KEY].astype(str)

cluster_summary = (
    obs_df.groupby([CLUSTER_KEY, annotation_col, lineage_col, confidence_col], observed=True)
    .size()
    .reset_index(name="n_cells")
)

cluster_summary["fraction_total_cells"] = cluster_summary["n_cells"] / adata.n_obs

# Add top category and diversity info
for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    cluster_summary = cluster_summary.merge(
        top_category_info(obs_df, CLUSTER_KEY, cat),
        on=CLUSTER_KEY,
        how="left"
    )
    cluster_summary = cluster_summary.merge(
        n_unique_by_cluster(obs_df, CLUSTER_KEY, cat),
        on=CLUSTER_KEY,
        how="left"
    )

# Entropy and global enrichment
for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    global_frac = obs_df[cat].value_counts(normalize=True).to_dict()
    ent_rows = []
    for cl, sub in obs_df.groupby(CLUSTER_KEY, observed=True):
        counts = sub[cat].value_counts()
        top_cat = counts.index[0]
        top_frac = counts.iloc[0] / counts.sum()
        global_top_frac = global_frac.get(top_cat, np.nan)
        enrich = top_frac / global_top_frac if global_top_frac and global_top_frac > 0 else np.nan
        ent_rows.append({
            CLUSTER_KEY: str(cl),
            f"{cat}_entropy": entropy_from_counts(counts.values),
            f"{cat}_norm_entropy": normalized_entropy_from_counts(counts.values),
            f"top_{cat}_global_fraction": global_top_frac,
            f"top_{cat}_enrichment_vs_global": enrich,
        })
    ent_df = pd.DataFrame(ent_rows)
    cluster_summary = cluster_summary.merge(ent_df, on=CLUSTER_KEY, how="left")

cluster_summary = cluster_summary.sort_values(
    CLUSTER_KEY,
    key=lambda s: s.astype(str).map(lambda x: int(x) if str(x).isdigit() else str(x))
).reset_index(drop=True)

write_tsv_replace(cluster_summary, CLUSTER_SUMMARY_TSV)
display(cluster_summary)


In [ ]:
# =========================
# Long-format composition table
# =========================

composition_parts = []

for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    tab = (
        obs_df.groupby([CLUSTER_KEY, cat], observed=True)
        .size()
        .reset_index(name="n_cells")
    )
    tab["variable"] = cat
    tab = tab.rename(columns={cat: "category"})
    tab["cluster_total_cells"] = tab.groupby(CLUSTER_KEY)["n_cells"].transform("sum")
    tab["fraction_within_cluster"] = tab["n_cells"] / tab["cluster_total_cells"]
    composition_parts.append(tab[[CLUSTER_KEY, "variable", "category", "n_cells", "cluster_total_cells", "fraction_within_cluster"]])

composition_long = pd.concat(composition_parts, axis=0, ignore_index=True)
composition_long = composition_long.sort_values([CLUSTER_KEY, "variable", "fraction_within_cluster"], ascending=[True, True, False])

write_tsv_replace(composition_long, COMPOSITION_LONG_TSV)
display(composition_long.head(30))


In [ ]:
# =========================
# Bias flags, review priorities, and CNV-role suggestions
# =========================

def contains_any(text, terms):
    text = str(text).lower()
    return any(t.lower() in text for t in terms)


def build_bias_reason(row):
    reasons = []

    n_cells = int(row["n_cells"])
    ann = str(row[annotation_col]).lower()
    lin = str(row[lineage_col]).lower()
    conf = str(row[confidence_col]).lower()

    if n_cells < 200:
        reasons.append("very_small_cluster_lt200")
    elif n_cells < 500:
        reasons.append("small_cluster_lt500")

    if row.get("top_dataset_id_fraction", 0) >= 0.90 and row.get("top_dataset_id_enrichment_vs_global", 0) >= 1.20:
        reasons.append("dataset_dominant")
    if row.get("top_condition_fraction", 0) >= 0.85 and row.get("top_condition_enrichment_vs_global", 0) >= 1.30:
        reasons.append("condition_dominant")
    if row.get("top_patient_id_fraction", 0) >= 0.70:
        reasons.append("patient_dominant")
    if row.get("top_sample_id_fraction", 0) >= 0.70:
        reasons.append("sample_dominant")

    if contains_any(ann + " " + lin + " " + conf, ["ambiguous", "mixed", "review", "ambient", "erythroid", "hb_high", "moderate_low", "low"]):
        reasons.append("annotation_or_marker_ambiguity")

    if contains_any(ann + " " + lin, ["putative_tumor", "tumor"]):
        reasons.append("malignancy_requires_CNV_confirmation")

    if len(reasons) == 0:
        reasons.append("no_major_bias_flag")

    return ";".join(reasons)


def assign_bias_status(reason):
    if "very_small_cluster_lt200" in reason or "erythroid" in reason or "ambient" in reason:
        return "review_or_exclude"
    if "annotation_or_marker_ambiguity" in reason:
        return "review_ambiguous"
    if "patient_dominant" in reason or "sample_dominant" in reason:
        return "patient_or_sample_dominant"
    if "dataset_dominant" in reason:
        return "dataset_dominant"
    if "condition_dominant" in reason:
        return "condition_dominant"
    return "broadly_represented"


def assign_review_priority(row):
    reason = str(row["cluster_bias_reason"])
    ann = str(row[annotation_col]).lower()
    n_cells = int(row["n_cells"])

    if "review_or_exclude" in row["cluster_bias_status"]:
        return "high"
    if contains_any(reason + " " + ann, ["ambiguous", "mixed", "ambient", "erythroid", "patient_dominant", "sample_dominant"]):
        return "high"
    if n_cells < 500 or contains_any(reason, ["dataset_dominant", "condition_dominant"]):
        return "medium"
    if contains_any(reason, ["malignancy_requires_CNV_confirmation"]):
        return "medium"
    return "standard"


def assign_cnv_role(row):
    ann = str(row[annotation_col]).lower()
    lin = str(row[lineage_col]).lower()
    conf = str(row[confidence_col]).lower()
    combined = " ".join([ann, lin, conf, str(row["cluster_bias_reason"]).lower()])

    if contains_any(combined, ["erythroid", "hb_high", "ambient", "mixed"]):
        return "exclude_or_review_before_CNV"
    if contains_any(combined, ["putative_tumor", "tumor"]):
        return "CNV_query_putative_tumor_like"
    if contains_any(combined, ["cycling"]):
        return "CNV_context_dependent_cycling_review"
    if contains_any(combined, ["oligodendrocyte", "endothelial", "t_nk", "b_plasma", "mast"]):
        return "CNV_reference_core_or_extended_normal_candidate"
    if contains_any(combined, ["myeloid", "tam", "monocyte", "macrophage"]):
        return "CNV_reference_extended_myeloid_candidate"
    if contains_any(combined, ["mural", "pericyte", "fibroblast", "stromal"]):
        return "CNV_reference_extended_stromal_review"
    return "CNV_role_review_needed"


bias_df = cluster_summary.copy()
bias_df["cluster_bias_reason"] = bias_df.apply(build_bias_reason, axis=1)
bias_df["cluster_bias_status"] = bias_df["cluster_bias_reason"].map(assign_bias_status)
bias_df["cluster_review_priority"] = bias_df.apply(assign_review_priority, axis=1)
bias_df["cnv_role_suggestion"] = bias_df.apply(assign_cnv_role, axis=1)

bias_cols = [
    CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "n_cells", "fraction_total_cells",
    "cluster_bias_status", "cluster_bias_reason", "cluster_review_priority", "cnv_role_suggestion",
    "top_dataset_id", "top_dataset_id_fraction", "top_dataset_id_enrichment_vs_global",
    "top_condition", "top_condition_fraction", "top_condition_enrichment_vs_global",
    "top_patient_id", "top_patient_id_fraction",
    "top_sample_id", "top_sample_id_fraction",
    "n_dataset_id", "n_condition", "n_patient_id", "n_sample_id",
]
bias_cols = [c for c in bias_cols if c in bias_df.columns]
bias_out = bias_df[bias_cols].copy()

write_tsv_replace(bias_out, BIAS_FLAGS_TSV)

cnv_role = bias_out[[CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "n_cells", "cnv_role_suggestion", "cluster_review_priority", "cluster_bias_status", "cluster_bias_reason"]].copy()
write_tsv_replace(cnv_role, CNV_ROLE_TSV)

cell_counts_by_status = (
    bias_out.groupby(["cluster_bias_status", "cluster_review_priority", "cnv_role_suggestion"], observed=True)["n_cells"]
    .sum()
    .reset_index()
    .sort_values("n_cells", ascending=False)
)
cell_counts_by_status["fraction_total_cells"] = cell_counts_by_status["n_cells"] / adata.n_obs
write_tsv_replace(cell_counts_by_status, CELL_COUNTS_BY_STATUS_TSV)

display(bias_out)
display(cell_counts_by_status)


In [ ]:
# =========================
# Add cluster-level bias/CNV labels to adata.obs
# =========================

BIAS_STATUS_COL = f"cluster_bias_status_{VERSION_TAG}"
BIAS_REASON_COL = f"cluster_bias_reason_{VERSION_TAG}"
REVIEW_PRIORITY_COL = f"cluster_review_priority_{VERSION_TAG}"
CNV_ROLE_COL = f"cnv_role_suggestion_{VERSION_TAG}"

bias_map = dict(zip(bias_out[CLUSTER_KEY].astype(str), bias_out["cluster_bias_status"]))
reason_map = dict(zip(bias_out[CLUSTER_KEY].astype(str), bias_out["cluster_bias_reason"]))
priority_map = dict(zip(bias_out[CLUSTER_KEY].astype(str), bias_out["cluster_review_priority"]))
cnv_map = dict(zip(bias_out[CLUSTER_KEY].astype(str), bias_out["cnv_role_suggestion"]))

adata.obs[BIAS_STATUS_COL] = adata.obs[CLUSTER_KEY].astype(str).map(bias_map).fillna("unknown")
adata.obs[BIAS_REASON_COL] = adata.obs[CLUSTER_KEY].astype(str).map(reason_map).fillna("unknown")
adata.obs[REVIEW_PRIORITY_COL] = adata.obs[CLUSTER_KEY].astype(str).map(priority_map).fillna("unknown")
adata.obs[CNV_ROLE_COL] = adata.obs[CLUSTER_KEY].astype(str).map(cnv_map).fillna("unknown")

# Simple aliases for downstream notebooks
adata.obs["cluster_bias_status"] = adata.obs[BIAS_STATUS_COL].astype(str)
adata.obs["cluster_review_priority"] = adata.obs[REVIEW_PRIORITY_COL].astype(str)
adata.obs["cnv_role_suggestion"] = adata.obs[CNV_ROLE_COL].astype(str)

adata.uns[f"annotation_bias_qc_{VERSION_TAG}"] = {
    "cluster_key": CLUSTER_KEY,
    "annotation_col": annotation_col,
    "lineage_col": lineage_col,
    "confidence_col": confidence_col,
    "bias_status_col": BIAS_STATUS_COL,
    "cnv_role_col": CNV_ROLE_COL,
    "main_purpose": "cluster composition, bias flags, and CNV role suggestions before CNV preparation",
}

print(adata.obs[[CLUSTER_KEY, annotation_col, BIAS_STATUS_COL, REVIEW_PRIORITY_COL, CNV_ROLE_COL]].head())


In [ ]:
# =========================
# Decision report
# =========================

decision_rows = []

for _, row in bias_out.iterrows():
    cl = str(row[CLUSTER_KEY])
    ann = str(row[annotation_col])
    lin = str(row[lineage_col])
    conf = str(row[confidence_col])
    status = str(row["cluster_bias_status"])
    priority = str(row["cluster_review_priority"])
    cnv_role = str(row["cnv_role_suggestion"])
    reason = str(row["cluster_bias_reason"])
    n_cells = int(row["n_cells"])

    if cnv_role == "CNV_query_putative_tumor_like":
        decision = "Keep as putative tumor-like query; do not call malignant until CNV confirms."
    elif cnv_role == "CNV_context_dependent_cycling_review":
        decision = "Keep as cycling review cluster; determine tumor/immune origin using CNV and markers."
    elif cnv_role.startswith("CNV_reference"):
        decision = "Candidate non-malignant reference/extended reference for CNV, subject to bias checks."
    elif cnv_role == "exclude_or_review_before_CNV":
        decision = "Review carefully; likely exclude from primary CNV reference/query if ambient/mixed/RBC-like."
    else:
        decision = "Review before assigning downstream CNV role."

    decision_rows.append({
        "cluster": cl,
        "n_cells": n_cells,
        "annotation": ann,
        "lineage": lin,
        "confidence": conf,
        "cluster_bias_status": status,
        "cluster_review_priority": priority,
        "cnv_role_suggestion": cnv_role,
        "cluster_bias_reason": reason,
        "decision": decision,
    })

decision_report = pd.DataFrame(decision_rows)
decision_report = decision_report.sort_values(
    "cluster",
    key=lambda s: s.astype(str).map(lambda x: int(x) if str(x).isdigit() else str(x))
)

write_tsv_replace(decision_report, DECISION_REPORT_TSV)
display(decision_report)


In [ ]:
# =========================
# QC figures: UMAPs
# =========================

if "X_umap" in adata.obsm:
    color_cols = [
        CLUSTER_KEY,
        annotation_col,
        lineage_col,
        confidence_col,
        BIAS_STATUS_COL,
        REVIEW_PRIORITY_COL,
        CNV_ROLE_COL,
        "core_dataset",
        "condition",
    ]
    color_cols = [c for c in color_cols if c in adata.obs.columns]

    for col in color_cols:
        try:
            legend_loc = "on data" if col == CLUSTER_KEY else "right margin"
            sc.pl.umap(
                adata,
                color=col,
                show=False,
                legend_loc=legend_loc,
                frameon=False,
                size=2,
                title=col,
            )
            out = FIG_DIR / f"umap_{col}.png"
            plt.savefig(out, dpi=180, bbox_inches="tight")
            plt.close()
            print("[SAVED]", out)
        except Exception as e:
            print("[WARNING] Could not plot UMAP for", col, repr(e))
else:
    print("[WARNING] X_umap not found; skipping UMAP plots.")


In [ ]:
# =========================
# QC figures: composition heatmaps
# =========================

def plot_cluster_category_fraction_heatmap(obs, cluster_col, category_col, out_png, title=None):
    tmp = (
        obs.groupby([cluster_col, category_col], observed=True)
        .size()
        .reset_index(name="n")
    )
    tmp["total"] = tmp.groupby(cluster_col)["n"].transform("sum")
    tmp["fraction"] = tmp["n"] / tmp["total"]

    mat = tmp.pivot(index=cluster_col, columns=category_col, values="fraction").fillna(0.0)

    # Sort numeric cluster labels if possible
    mat = mat.loc[sorted(mat.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]

    fig_w = max(8, 0.6 * mat.shape[1] + 4)
    fig_h = max(5, 0.35 * mat.shape[0] + 2)

    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(mat.values, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
    plt.colorbar(label="Fraction within cluster")
    plt.xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    plt.yticks(range(mat.shape[0]), mat.index)
    plt.xlabel(category_col)
    plt.ylabel(cluster_col)
    plt.title(title or f"{category_col} composition per cluster")
    plt.tight_layout()
    plt.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close()
    print("[SAVED]", out_png)


plot_cluster_category_fraction_heatmap(
    adata.obs,
    CLUSTER_KEY,
    "dataset_id",
    FIG_DIR / f"heatmap_dataset_id_composition_{VERSION_TAG}.png",
    title=f"Dataset composition per cluster — {VERSION_TAG}",
)

plot_cluster_category_fraction_heatmap(
    adata.obs,
    CLUSTER_KEY,
    "condition",
    FIG_DIR / f"heatmap_condition_composition_{VERSION_TAG}.png",
    title=f"Condition composition per cluster — {VERSION_TAG}",
)

plot_cluster_category_fraction_heatmap(
    adata.obs,
    CLUSTER_KEY,
    BIAS_STATUS_COL,
    FIG_DIR / f"heatmap_bias_status_by_cluster_{VERSION_TAG}.png",
    title=f"Bias status per cluster — {VERSION_TAG}",
)


In [ ]:
# =========================
# Save h5ad for next notebooks
# =========================

# Remove old rank_genes_groups objects if present to keep h5ad smaller and avoid serialization surprises.
for k in list(adata.uns.keys()):
    if str(k).startswith("rank_genes_groups"):
        del adata.uns[k]
        print("[REMOVED old uns key]", k)

write_h5ad_replace(adata, OUT_H5AD)
write_h5ad_replace(adata, OUT_ALIAS_H5AD)

print("Final object:")
print(adata)


In [ ]:
# =========================
# Output manifest
# =========================

manifest_items = [
    ("input_h5ad", ANNOTATED_H5AD),
    ("output_h5ad", OUT_H5AD),
    ("output_alias_h5ad", OUT_ALIAS_H5AD),
    ("readiness_report", READINESS_TSV),
    ("cluster_summary", CLUSTER_SUMMARY_TSV),
    ("composition_long", COMPOSITION_LONG_TSV),
    ("bias_flags", BIAS_FLAGS_TSV),
    ("cnv_role_recommendations", CNV_ROLE_TSV),
    ("cell_counts_by_status", CELL_COUNTS_BY_STATUS_TSV),
    ("decision_report", DECISION_REPORT_TSV),
    ("figures_dir", FIG_DIR),
]

manifest = pd.DataFrame([
    {
        "item": name,
        "path": str(path),
        "exists": Path(path).exists() if not str(path).endswith(str(FIG_DIR)) else Path(path).exists(),
    }
    for name, path in manifest_items
])

write_tsv_replace(manifest, OUTPUT_MANIFEST_TSV)
display(manifest)
